In [1]:
import re
import spacy
nlp = spacy.load("en_core_web_md")
# nlp = spacy.load("en_core_web_lg")
POS_MAP = {
    "VERB": "V",   # Verb: 动词（如: run, eat, situated）
    "AUX": "V",    # Auxiliary: 助动词（如: is, has, should，统一映射为V简化识别）
    "ADP": "P",    # Adposition: 介词（如: in, on, at, among）
    "NOUN": "N",   # Noun: 普通名词（如: car, court, field）
    "PROPN":"N",   # Proper Noun: 专有名词（如: London, image，统一映射为N）
    "PRON": "R",   # Pronoun: 代词（如: it, they, there, which）
    "DET":  "D",   # Determiner: 限定词（如: the, a, this, both）
    "ADJ":  "J",   # Adjective: 形容词（如: small, rectangular, bottom-most）
    "ADV":  "Q",   # Adverb: 副词（如: very, well, extremely, most）
    "CCONJ":"C",   # Coordinating Conjunction: 并列连词（如: and, or, but）
    "SCONJ":"S",   # Subordinating Conjunction: 从属连词（如: if, because, although）
    "PART":  "T",  # Particle: 助词（如: 's, not, to-inf）
    "PUNCT":"X",   # Punctuation: 标点符号（如: , . - ，用于处理连字符）
    "NUM":  "M",   # Numeral: 数词（如: one, 1, three）
    "SYM":  "Y",   # Symbol: 符号（如: $, %, +）
    "INTJ": "I",   # Interjection: 感叹词（如: hello, wow）
}

In [3]:

#nlp这个库用的中等版本，有时候识别词性可能会出问题
noun_words = {'marking', 'sidewalk'}
direction_words = {'left', 'right', 'top', 'bottom', 'front', 'back'}
VALID_START_TAGS = {'N', 'D', 'J','R','M'}

def split_by_verb_prep_custom(text):
    text = ' '.join(text.split())
    sub_texts = [s.strip() for s in text.split(',')]
    extracted_np, final_subs, found = None, [], False
    for sub in sub_texts:
        if found:
            final_subs.append(sub)
            continue
        doc = nlp(sub)
        # 标签生成：注意这里把连字符单独映射为 L
        tags = ["N" if t.text.lower() in noun_words else
                "J" if t.text.lower() in direction_words else
                "L" if t.text == '-' else
                POS_MAP.get(t.pos_, "O") for t in doc]
        pos_seq = "".join(tags)

        # 判定条件：单句直接进，多句则首词必须符合白名单
        if len(sub_texts) == 1 or (pos_seq and pos_seq[0] in VALID_START_TAGS):
            # --- 正则修复：[CJQLX]* 是修饰前缀，N+(?:LN+)* 兼容单名词和复合名词 ---
            m = re.search(r'[CJQLX]*N+(?:LN+)*', pos_seq)
            if m:
                t_starts = [t.idx for t in doc] + [len(sub)]

                # 往前看一位，把冠词 "The/A" 也抓进 extracted_np
                full_start = m.start()
                extracted_np = sub[t_starts[full_start]:t_starts[m.end()]].strip()

                # --- 替换修复：只替换最后的名词簇，防止插入到 right-most 中间 ---
                all_n_matches = list(re.finditer(r'N+(?:LN+)*', pos_seq[m.start():m.end()]))
                if all_n_matches:
                    last_n = all_n_matches[-1]
                    ns = m.start() + last_n.start()
                    ne = m.start() + last_n.end()

                    prefix = sub[:t_starts[ns]].rstrip()
                    suffix = sub[t_starts[ne]:].lstrip()
                    final_subs.append(f"{prefix} something {suffix}".strip())
                    found = True
                    continue
        final_subs.append(sub)

    return [extracted_np, ", ".join(final_subs)] if extracted_np else [text, "something"]

test_sentences = [
    # 简单结构
    # 'The right-most tennis court on the left side of the image.',
    # "Located at the bottom of the dock, the ship is perpendicular to the pier.",
    # "The top-right vehicle distinguished as the right-most vehicle on its road segment.",
    # "the tiny vehicle in the middle is on the lower left of the long gray ship at the bottom",
    # "The bottom-most windmill is positioned at the lower end of the image.  ",
    # "the roundabout on the left ",
    # "Among the small vehicles situated on the edge of the road closer to the bottom.",
    # "The swimming pool, with its rectangular shape, is located on the right side of the image.",
    # "Positioned at the bottom of the image.",
    # "The right-most basketball court of the image."
    # 
    # # 中等复杂度
    # "The rectangular soccer ball field located in the middle-left of the image.",
    # "The rightmost tennis court is adjacent to a grey surface resembling a hard court or parking lot.",
    # "The vehicle located at the top of the image.",
    # "The largest small vehicle is towards the middle right of the image.",
    # "The left-most small vehicle located in the bottom-right quarter.",
    # "The largest ship is positioned at the top-most part of the harbor area and has a noticeable elongated structure.",
    # "The baseball diamond is located at the bottom-middle part of the image.",
    # " One vehicle is positioned in the middle-right area.",
    # "The top-most small vehicle located towards the top edge of the image.",
    # "The left-most harbor section extends vertically across nearly the entire left side of the image.",
    # "In close proximity to the ground-track field, there is a small soccer-ball field situated towards the bottom-right of the image.",
    # "In the bottom-right corner, one storage tank is located.",
    # "The largest large-vehicle visible on the highway is situated at the bottom-left of the image.",
    # " The object visible with a clear, extended runway amidst the agricultural fields is a small airport."
    
]
for s in test_sentences:
    print(f"\n原句: {s}")
    w=split_by_verb_prep_custom(s)
    print(f"切分: {w}")


原句: The right-most tennis court on the left side of the image.
切分: ['right-most tennis court', 'The right-most something on the left side of the image.']

原句: Located at the bottom of the dock, the ship is perpendicular to the pier.
切分: ['ship', 'Located at the bottom of the dock, the something is perpendicular to the pier.']

原句: The top-right vehicle distinguished as the right-most vehicle on its road segment.
切分: ['top-right vehicle', 'The top-right something distinguished as the right-most vehicle on its road segment.']

原句: the tiny vehicle in the middle is on the lower left of the long gray ship at the bottom
切分: ['tiny vehicle', 'the tiny something in the middle is on the lower left of the long gray ship at the bottom']

原句: The bottom-most windmill is positioned at the lower end of the image.  
切分: ['bottom-most windmill', 'The bottom-most something is positioned at the lower end of the image.']

原句: the roundabout on the left 
切分: ['roundabout', 'the something on the left']

